# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check A -- staleness (behind the refresh flags)

**Claim to check:** pages that haven't been updated in a while are more likely to be declining.

Bucket table: `freshness_tier` vs. decline rate (`trend_direction == 'down'`) and mean `trend_pct`.
Base rate for comparison: dataset-wide decline rate is **54.2%**.

| freshness_tier | n | decline_rate | mean_trend_pct |
|---|---|---|---|
| 0-30   | 20,480 | 0.511 | +0.78 |
| 31-90  | 175    | 0.589 | -7.37 |
| 91-180 | 9,171  | 0.611 | -15.68 |
| 181+   | 174    | 0.471 | -6.78 |

**Verdict: MIXED.** Across the two dominant buckets (0-30 -> 91-180, together 98.8% of rows) staleness
tracks decline cleanly: decline rate climbs from 51.1% to 61.1% and mean trend goes from slightly
positive to -15.7%. But the pattern **inverts** at the most-stale tier (181+): decline rate drops back
below the fresh-page rate, on only 174 rows. That's too little data to trust, so this signal is real
for the *91-180* band specifically, not for "the older the worse" in general. The rule below uses
`freshness_tier == '91-180'`, not an open-ended `days_since_last_update >= X` threshold, on purpose.

### Signal check B -- CTR vs. position (behind the CTR-fix logic)

**Claim to check:** CTR should track position -- better ranking pages should get more clicks per
impression.

Bucket table: `position_tier` vs. CTR, with the volume floor the data dictionary explicitly warns about.

| position_tier | n | mean_ctr | median_ctr | median_impressions_90d |
|---|---|---|---|---|
| top_3     | 2,321  | 1.48 | 0.00 | 3.0    |
| page_1    | 11,814 | 0.65 | 0.16 | 1,179.5 |
| striking  | 7,304  | 0.32 | 0.11 | 874.5  |
| page_3_5  | 7,242  | 0.22 | 0.03 | 811.5  |
| deep      | 1,319  | 0.15 | 0.00 | 218.0  |

**Verdict: CONFIRMED, with a caveat.** CTR declines monotonically as position gets worse across all
five tiers -- exactly the expected relationship, and the direction a CTR-fix flag should rely on. The
caveat: `top_3`'s median volume is only **3 impressions/90d**, so its higher mean CTR is mostly noise
from tiny denominators (one click on 3 impressions is a 33% CTR). A CTR-fix rule that only looks at
`position_tier` without a volume floor would chase noise in the `top_3` bucket. That's why the rule
below requires `impressions_90d >= 300` regardless of which signal triggered it.

### The rule, in plain words

A page is worth a refresh nudge if it hasn't been touched in 91-180 days (the band where staleness
actually tracked decline above) **and** it still has real visible traffic (>= 300 impressions in 90
days, clearing the volume floor from Signal B). Score it by how much traffic is at stake, so the
biggest opportunities surface first.

**Reason code:** `stale_but_visible`
**Action label:** `refresh_page` if the rule fires, else `no_action`


In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

stale = (df['freshness_tier'] == '91-180').astype(int)
visible = (df['impressions_90d'] >= 300).astype(int)

# --- Signal check A: staleness vs decline (verify before trusting) ---
sig_a = df.groupby('freshness_tier', observed=True).agg(
    n=('content_id', 'size'),
    decline_rate=('trend_direction', lambda s: (s == 'down').mean()),
    mean_trend_pct=('trend_pct', 'mean'),
).reindex(['0-30', '31-90', '91-180', '181+'])
print("Signal A -- freshness_tier vs decline (base rate = {:.3f})".format((df['trend_direction'] == 'down').mean()))
print(sig_a)
print()

# --- Signal check B: CTR vs position, with volume floor ---
sig_b = df.groupby('position_tier', observed=True).agg(
    n=('content_id', 'size'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    median_impressions_90d=('impressions_90d', 'median'),
).reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
print("Signal B -- CTR by position_tier (volume floor check)")
print(sig_b)
print()

# --- The rule itself: readable on purpose, no fitted weights, no label inputs ---
df['score'] = stale * visible * df['impressions_90d']
df['reason_code'] = 'stale_but_visible'
df['action'] = df['score'].apply(lambda s: 'refresh_page' if s > 0 else 'no_action')

print("Rows flagged for action:", (df['action'] == 'refresh_page').sum(), "/", len(df))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

No new signals here -- this cell just ranks by the score already computed above and writes the
CSV the CI leak-guard expects to find at `work/outputs/`. `content_id` / `client_id` ride along for
identification only, never as score inputs (per `flyrank-data/SKILL.md`: IDs are pseudonyms, joins
and grouping only).

In [ ]:
import os

queue = df.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
queue[['content_id', 'client_id', 'score', 'reason_code', 'action']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False
)

print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print("Flagged (refresh_page):", (queue['action'] == 'refresh_page').sum())
queue.head(10)[['content_id', 'client_id', 'score', 'reason_code', 'action']]


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

> Note: this skeleton's own header says "top-20" (leftover from the template's default); the
> assignment card for this week asks for a **top-10** review, so that's what's done here.

For each, `position_tier`/`avg_position`/`ctr`/`trend_pct`/`trend_direction` are pulled in as
**diagnostics for the human review only** -- they were not part of the score, and looking at them
after the fact to sanity-check a pick is not leakage; feeding them into the score formula would be.

1. **content_5fe46e04994d** (score 517,715, page_1 @ 4.2, CTR 0.14%, trend -44.8% down) -- the single
   largest page in the whole dataset happens to be stale; the real decline supports the pick.
   **Wrong if:** the 0.14% CTR (well under the page_1 average of 0.65%) is actually a title/snippet
   problem, not a staleness problem -- refreshing the body content wouldn't fix that.
2. **content_2dba2b1f9536** (score 443,434, page_3_5 @ 27.9, CTR 0.21%, trend +1.4% stable) -- flagged
   purely on staleness + volume; trend is stable, not declining.
   **Wrong if:** a stable page doesn't need fixing at all -- the rule doesn't check trend, so it
   surfaces pages that aren't actually in trouble.
3. **content_2c2606c5d176** (score 347,399, page_1 @ 4.2, CTR 0.53%, trend -36.5% down) -- CTR is
   close to the page_1 tier average, real decline, commercial intent.
   **Wrong if:** the decline is external (a competitor or SERP feature change) rather than content
   quality -- a refresh wouldn't reverse a competitive shift.
4. **content_cb112fce36be** (score 309,910, page_1 @ 5.6, CTR 0.16%, trend -41.8% down) -- CTR well
   under the page_1 tier average again.
   **Wrong if:** the low CTR, not the staleness, is the real lever -- a snippet/title fix might work
   faster than a content refresh.
5. **content_9532f197bbc8** (score 309,192, top_3 @ 2.0, CTR 0.87%, trend -37.3% down) -- an excellent
   position with above-average CTR for its tier, but declining anyway.
   **Wrong if:** this is a temporary ranking fluctuation on an otherwise strong evergreen page --
   refreshing something that's already working risks breaking what isn't broken.
6. **content_36ff89c8214e** (score 295,097, page_1 @ 7.3, CTR 0.05%, trend +0.5% stable) -- tiny CTR,
   but trend is stable, not falling.
   **Wrong if:** the low CTR is structural for this query type (e.g. a branded/navigational query with
   naturally low click intent) rather than a fixable content problem.
7. **content_b28d1efd668f** (score 286,608, page_3_5 @ 26.2, CTR 0.06%, trend -17.2% stable) -- decent
   traffic at stake but position is deep.
   **Wrong if:** at position 26, the real blocker is authority/backlinks, not content freshness -- a
   refresh alone rarely moves a page from page 3 to page 1.
8. **content_813e88069237** (score 233,561, page_3_5 @ 26.2, CTR 0.06%, trend -33.8% down) -- same
   position band as #7, but genuinely declining.
   **Wrong if:** same as #7 -- a refresh without added depth/links likely won't move a page-3 result.
9. **content_c21024970297** (score 211,366, page_1 @ 5.1, CTR 0.41%, trend -11.6% stable) -- CTR near
   the page_1 average, trend stable, not declining.
   **Wrong if:** stable + already-decent CTR means there's no real problem to fix here -- another rule
   pick that isn't in trouble yet.
10. **content_c8e9d6ab9013** (score 208,678, page_1 @ 9.7, CTR 0.00%, trend -43.4% down) -- 0% CTR at
    the edge of page 1 is unusual.
    **Wrong if:** a literal 0% CTR points to a technical issue (broken canonical, accidental noindex,
    a misleading snippet) rather than staleness -- a content refresh wouldn't fix a technical block.


In [ ]:
cols = ['content_id', 'score', 'position_tier', 'avg_position', 'ctr',
        'content_type', 'main_intent', 'trend_pct', 'trend_direction']
queue_with_diagnostics = df.sort_values('score', ascending=False).reset_index(drop=True)
queue_with_diagnostics.head(10)[cols]


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks, found on purpose (skill's own bar: "if it found none, look harder"):**

- **4 of the top 10** (#2, #6, #7, #9) have `trend_direction != 'down'` -- the rule flags them on
  staleness + volume alone and doesn't check whether the page is actually declining. That's a real
  gap: roughly 40% of the top of this queue isn't currently in trouble.
- **The score itself is outlier-dominated.** Raw `impressions_90d` as the multiplier means the top
  of the queue is really "biggest stale pages," not "best refresh opportunities" -- row 1 alone
  carries 517,715 impressions against a dataset median of 731. A log-scaled score would spread the
  ranking out instead of being led by a handful of power-law outliers.
- **Informal lift check:** the flagged set (`refresh_page`, n=7,212) declines 61.7% of the time vs.
  51.8% for everything else (base rate 54.2%). That's a real but modest lift -- not a rule that's
  obviously right most of the time, which is exactly why the model weeks need to beat it.

**Leakage check:** the score uses only `freshness_tier` (from `days_since_last_update`) and
`impressions_90d` -- neither is `trend_direction`, `trend_pct`, `impressions_last_30d`, or
`impressions_prev_30d`. Per `docs/data-dictionary.md`, `trend_direction`/`trend_pct` are the literal
source of the label and are never touched here. `impressions_90d` is a trailing 90-day total already
observed at export time, not a future window. No product flags or future-window inputs went into the
score.

In [ ]:
leak_cols = {'trend_direction', 'trend_pct', 'is_declining_label',
             'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d'}
score_inputs = {'freshness_tier', 'impressions_90d'}

assert score_inputs.isdisjoint(leak_cols), "score touched a label-derived or future-window column"
print("Leakage check passed -- score inputs:", score_inputs)

# Weak-pick evidence: informal lift vs base rate (diagnostic only, not part of the rule)
flagged = df['action'] == 'refresh_page'
print("decline rate among flagged:    {:.3f}".format(df.loc[flagged, 'trend_direction'].eq('down').mean()))
print("decline rate among unflagged:  {:.3f}".format(df.loc[~flagged, 'trend_direction'].eq('down').mean()))
print("dataset base rate:             {:.3f}".format(df['trend_direction'].eq('down').mean()))


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

> Numbers above were computed for real against `data/raw/content_refresh_anonymized.csv` (a public file in this repo) before this notebook was written -- they are not placeholders. Still run Runtime -> Run all yourself in Colab to regenerate `work/outputs/baseline_action_score.csv` and confirm it reproduces the same numbers, then check the remaining boxes.